In [ ]:
import json

try :
    with open("questions.json","r",encoding="utf-8") as file:
        test_set = json.load(fp=file)
except FileNotFoundError:
    print("Error : file questions.json does not exist ")
except json.JSONDecodeError:
    print("Error: failed to pasrse json")

print(test_set)

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_openai.embeddings import OpenAIEmbeddings

docs = PyPDFLoader(file_path="/Users/sanvijanvi/Documents/SJ_HomeWork/Novatech_HR_Policy_Handbook_2026.pdf").load()
embed = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

        # hit3+= any(p for p in pages[:3] if p in case["expected_pages"]) <br>
        # hit5+= any(p for p in pages[:5] if p in case["expected_pages"])

        # pages = [0, 5, 6]
        # expected = [0]

        # any(p for p in pages if p in expected)    # receives 0 → False  ❌ (wrong)
        # any(p in expected for p in pages)          # receives True → True ✅

# evalaute retrieval

In [ ]:
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

def build_retriever(embed, chunk_size,chunk_overlap,docs,k=5):
    splitter = RecursiveCharacterTextSplitter(chunk_size = chunk_size,chunk_overlap = chunk_overlap)
    chunks = splitter.split_documents(docs)
    vector_store = FAISS.from_documents(documents= chunks, embedding=embed)
    return vector_store.as_retriever(search_kwargs = {"k":k}),len(chunks)

def evaluate_retrieval(retriever, testSet):
    hit3 = hit5 = rr_totoal = 0
    for case in testSet :
        results = retriever.invoke(case["question"])
        pages = [d.metadata["page"]+1 for d in results] #d.metadata["page"]+1  , bcoz pypdf pages start from 0

        if not any(p in case["expected_pages"] for p in pages[:3]):
            print(case["id"], case["type"], case["question"], "expected", case["expected_pages"], "got", pages)
       
        hit3+= any(p for p in case["expected_pages"] if p in pages[:3])
        hit5+= any(p for p in case["expected_pages"] if p in pages[:5])

        ranks = [i+1 for i, p in enumerate(pages) if p in case["expected_pages"]]
        rr_totoal += 1/ranks[0] if ranks else 0

        # print(pages, hit3, hit5, ranks,f"mrr =",round(rr_totoal,3) )

    n = len(testSet)
    return {"hit3":hit3/n,"hit5": hit5/n, "mmr":round(rr_totoal/len(testSet),3)}


In [ ]:
for size, overlap in [(300,30),(500,50),(1000,100)]:
    retriever, chunkLen = build_retriever(embed, size, overlap,docs)
    print(f"size , overlap, chunksLength = {size},{overlap},{chunkLen}:",evaluate_retrieval(retriever,test_set))

size , overlap, chunksLength = 300,30,74: {'hit3': 0.9666666666666667, 'hit5': 1.0, 'mmr': 0.886}<br>
size , overlap, chunksLength = 500,50,42: {'hit3': 0.9666666666666667, 'hit5': 0.9666666666666667, 'mmr': 0.861}<br>
size , overlap, chunksLength = 1000,100,23: {'hit3': 0.9333333333333333, 'hit5': 1.0, 'mmr': 0.861}<br>

Note : Chunk size and overlap -> 300/30 selected.

# embedding models 

In [ ]:
embedders = {
    "MiniLM (local)": HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2"),
    "bge-small (local)": HuggingFaceEmbeddings(
        model_name="BAAI/bge-small-en-v1.5",
        encode_kwargs={"normalize_embeddings": True},
    ),
    "OpenAI 3-small (API)": OpenAIEmbeddings(model="text-embedding-3-small"),
}

In [ ]:
for name, embed in embedders.items():
    retriever, _ = build_retriever(embed, 300, 30, docs)
    print(name, evaluate_retrieval(retriever, test_set))

# embedding models 

MiniLM (local) {'hit3': 0.9666666666666667, 'hit5': 1.0, 'mmr': 0.886}<br>
bge-small (local) {'hit3': 0.9333333333333333, 'hit5': 0.9666666666666667, 'mmr': 0.901}<br>
OpenAI 3-small (API) {'hit3': 0.9666666666666667, 'hit5': 1.0, 'mmr': 0.975}<br>

# ensembele retriever


Consider above best 2 embedding models<br>

bge-small (local) {'hit3': 0.9333333333333333, 'hit5': 0.9666666666666667, 'mmr': 0.901}<br>
OpenAI 3-small (API) {'hit3': 0.9666666666666667, 'hit5': 1.0, 'mmr': 0.975}<br>

In [ ]:
from langchain_classic.retrievers import BM25Retriever, EnsembleRetriever

In [ ]:
import re

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = splitter.split_documents(docs)
bm25_retriever = BM25Retriever.from_documents(documents=chunks,preprocess_func=lambda text: re.findall(r"\w+", text.lower()))
bm25_retriever.k = 5

In [ ]:
import re
for name, embed in embedders.items():
    faiss_retriever = FAISS.from_documents(chunks, embed).as_retriever(search_kwargs={"k": 5})
    ensemble_retriever = EnsembleRetriever(retrievers=[faiss_retriever,bm25_retriever],weights=[0.5,0.5])
    print(name, evaluate_retrieval(retriever=ensemble_retriever,testSet=test_set))

In [ ]:
evaluate_retrieval(retriever=bm25_retriever,testSet=test_set)

In [ ]:
final_embed = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5",encode_kwargs={"normalize_embeddings": True})
for size, overlap in [(300,30),(500,50)]:
    base_retriever, _= build_retriever(embed=final_embed,chunk_size=size, chunk_overlap= overlap, docs=docs)
    bm25_retriever = BM25Retriever.from_documents(documents=RecursiveCharacterTextSplitter(chunk_size = size, chunk_overlap =overlap).split_documents(docs),preprocess_func=lambda text: re.findall(r"\w+", text.lower()))
    bm25_retriever.k = 5
    ensemble_retriever = EnsembleRetriever(retrievers=[base_retriever,bm25_retriever],weights=[0.5,0.5])
    print(f"{size}/{overlap}", evaluate_retrieval(retriever=ensemble_retriever,testSet=test_set))

In [ ]:
from rag import get_rag

retriever = get_rag()

In [ ]:
from langchain_groq.chat_models import ChatGroq
from langchain_openai.chat_models import ChatOpenAI

In [ ]:
answer_llm = ChatGroq(model="qwen/qwen3.8-27b",temperature=0)
judge_llm  = ChatOpenAI(model="gpt-5.4-mini",temperature=0)

In [ ]:
answer_promt = """
    Answer the question {question} only using the HR policy context. context : {context}.
    If context does not know the answer , say you dont know.
"""

def generate_answer(question):
    context = retriever.invoke(question)
    answer = answer_llm.invoke(input=answer_promt.format(question = question, context = context))
    return answer, context

answer, context = generate_answer("Does the company help with daycare costs for my toddler?")

In [ ]:
answer.content

In [ ]:
len(context)

In [ ]:
for doc in context:
    print(doc.page_content)

In [ ]:
from pydantic import BaseModel, Field
class RagGrade(BaseModel):
    fairness : int = Field(description="1:5 = Is answer represents the given context?")
    correctness : int = Field(description="1:5 = Is answer similar to the given reference answer?")
    relevance : int = Field(description="1:5 = Is answer relevant to the given question?")
    reason : str = Field(description="Give reason in one line")

In [ ]:
judge_structured_llm = judge_llm.with_structured_output(RagGrade)

In [ ]:
import os, json

In [ ]:
str_dir = str_dir = os.getcwd()
json_path = os.path.join(str_dir,"questions.json")
with open(file = json_path,mode="r",encoding="utf-8") as f:
    test_set = json.load(f)

In [ ]:
judge_prompt = """
    You are grading the anwer for RAG system using following inputs

    Question : {question}
    Refernce : {reference}
    Context : {context}
    System_answer : {answer}
"""

In [ ]:
import time
import pandas as pd
def evalaute_generation(test_set):
    rows = []
    for case in test_set:
        print(case["id"])
        answer, context = generate_answer(question=case["question"])
        grade = judge_structured_llm.invoke(input= judge_prompt.format(question = case["question"], reference = case["reference"], context = context,answer=answer))
        rows.append({"id":case["id"],"type":case["type"],"question":case["question"],"answer":answer,"fairness":grade.fairness,"correctness":grade.correctness,
                     "relevance":grade.relevance,"reason":grade.reason})
        time.sleep(2)
    return pd.DataFrame(rows)

In [ ]:
df = evalaute_generation(test_set)

In [ ]:
print(df)

In [ ]:

df.to_csv(path_or_buf= os.path.join(str_dir,"data","generation_eval.csv"),index=False)
metrics = ["fairness", "relevance", "correctness"]
print(df[metrics].mean().round(2))
print(df.groupby("tyep")[metrics].mean().round(2))

# Agent evalaution

In [ ]:
from agent import getAgent

agent = getAgent()

In [ ]:
dataset = [
    {"id": 1, "category": "leave_balance",
     "question": "I've used 11 earned leaves this year. How many are left?",
     "expected_tools": ["leave_balance"],
     "expected_args": {"leave_balance": {"leave_type": "earned", "days_taken": 11}},
     "reference": "7 days remaining (18 - 11)"}
     ]

config = {"configurable":{"thread_id":"user123"}}

In [ ]:
from langchain_core.messages import AIMessage,ToolMessage

In [ ]:
def extract(messages):
    calls , outputTool =[], {}
    for msg in messages:
        if isinstance(msg,AIMessage):
            calls.extend(msg.tool_calls)
        elif isinstance(msg,ToolMessage):
            outputTool[msg.tool_call_id] = str(msg.content).lower()
    return calls, outputTool


In [ ]:
data = dataset[0]
messages = agent.invoke(input = {"messages":[{"role":"user","content":data["question"]}]},config=config)["messages"]

In [ ]:
calls , output = extract(messages)

In [ ]:
calls

In [ ]:
output

In [ ]:
count = dict(n_calls=0, hit_tools = 0, hit_arg = 0, hit_answer = 0, hit_message = 0)

In [ ]:
count["n_calls"] += len(calls)
count["n_calls"]

In [ ]:
exp = data["expected_tools"]
called_tools = [t["name"] for t in calls]
exp

In [ ]:
set(called_tools)

In [ ]:
if set(exp) <= set(called_tools):
    print("ALl called")
    count["hit_tools"] += 1

In [ ]:
#check args
dataset1= [
    {"id": 1, "category": "leave_balance",
     "question": "I've used 11 earned leaves this year. How many are left?",
     "expected_tools": ["leave_balance"],
     "expected_args": {"leave_balance": {"leave_type": "earned", "days_taken": 11}},
     "reference": "7 days remaining (18 - 11)"}, 
     {"id": 18, "category": "multi_tool",
     "question": "Is there a limit on how many casual leaves I can take in a row? Also, I've used 6 casual leaves, how many are left?",
     "expected_tools": ["search_hr_policy", "leave_balance"],
     "expected_args": {"leave_balance": {"leave_type": "casual", "days_taken": 6}},
     "reference": "Maximum 3 consecutive days; 2 casual leaves remaining"}
     ]

In [ ]:
calls

In [ ]:
[c["name"] for c in calls]

In [ ]:
def arg_check(name, args):
    print(name,"***",args)

any(arg_check(name, args) for name, args in data["expected_args"])